# Сборка датасета для модели сутяжности клиента

Пайплайн вынесен в пакет `querulus.dataset`. Тетрадка только настраивает окружение и вызывает `run_pipeline`.

## Настройка

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Корень репозитория и src/ — как в integration/tests/__init__.py
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import logging

import pandas as pd

from querulus.dataset import run_pipeline
from querulus.dataset.io import setup_notebook_logging

setup_notebook_logging(logging.INFO)

# True: подгрузить готовый финальный датасет; False: пересобрать пайплайн
LOAD_FINAL_DATASET = True
FINAL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
# None=авто; True=старый parquet; False=новый (пайплайн или icnl-датасет)
LOAD_LEGACY_DATASET = None

# False: parquet из data/raw (или SQL при отсутствии); True: всегда SQL
USE_SQL = False
SAVE_CHECKPOINT = True

# Таргеты в датасете (build_targets):
#   legacy (Litigant):  TARGET_2, TARGET_3_SEV
#   новые (icnl):       TARGET_FREQ, TARGET_SEV
# Для обучения:
#   FREQUENCY_TARGET = "TARGET_2" | "TARGET_FREQ"
#   SEVERITY_TARGET  = "TARGET_3_SEV" | "TARGET_SEV"
FREQUENCY_TARGET = "TARGET_FREQ"
SEVERITY_TARGET = "TARGET_SEV"

## Пайплайн

In [ ]:
if LOAD_FINAL_DATASET:
    if not FINAL_DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Финальный датасет не найден: {FINAL_DATASET_PATH}. "
            "Поставьте LOAD_FINAL_DATASET=False, чтобы собрать его заново."
        )
    df = pd.read_parquet(FINAL_DATASET_PATH)
    print(f"LOAD final dataset: {FINAL_DATASET_PATH} shape={df.shape}")
else:
    df = run_pipeline(use_sql=USE_SQL, save_checkpoint=SAVE_CHECKPOINT)

## Проверка результата

In [ ]:
print(f"frequency target: {FREQUENCY_TARGET}")
display(df[FREQUENCY_TARGET].value_counts(dropna=False))
print(f"severity target: {SEVERITY_TARGET}")
display(df[SEVERITY_TARGET].describe())

In [ ]:
# Проверка несогласованности: TARGET_2 == 0 при TARGET_3_SEV/TARGET_SEV > 0
sev_col = "TARGET_3_SEV" if "TARGET_3_SEV" in df.columns else "TARGET_SEV"

if "TARGET_2" not in df.columns:
    raise KeyError("Нет колонки TARGET_2. Пересоберите датасет: LOAD_FINAL_DATASET=False.")

target_2 = df["TARGET_2"].fillna(0)
mask = (target_2 == 0) & (df[sev_col].fillna(0) > 0)

print(f"Rows: {int(mask.sum())} / {len(df)}")
if "INCIDENT_NUMBER" in df.columns:
    print(f"Incidents: {df.loc[mask, 'INCIDENT_NUMBER'].nunique()}")

cols = [c for c in ["INCIDENT_NUMBER", "LOSS_NUMBER", "TARGET_2", sev_col] if c in df.columns]
display(df.loc[mask, cols].head(50))

## Сверка таргетов

`querulus.training.target_compare` — legacy vs new и Litigant vs Querulus.

In [ ]:
from querulus.training.target_compare import (
    compare_litigant_vs_querulus,
    compare_old_vs_new_targets,
    default_litigant_dataset_path,
)

old_vs_new = compare_old_vs_new_targets(df)
display(old_vs_new.report)

print("=== Топ расхождений: регрессия ===")
display(old_vs_new.top_regression)

print("=== Топ расхождений: классификация ===")
display(old_vs_new.top_classification)

litigant_path = default_litigant_dataset_path(PROJECT_ROOT)
if litigant_path.exists():
    lit_vs_qry = compare_litigant_vs_querulus(pd.read_parquet(litigant_path), df)
    display(lit_vs_qry.report)
    print("=== Litigant vs Querulus: регрессия ===")
    display(lit_vs_qry.top_regression)
else:
    print(f"Litigant parquet не найден: {litigant_path}")

## EDA признаков

`research_continous` / `research_feature` по MVP-признакам (до обучения, как в `model_learn.py`).

In [ ]:
from querulus.training.config import TrainingConfig
from querulus.training.plots import run_mvp_frequency_eda

EDA_CONFIG = TrainingConfig(
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
)
if "expos" not in df.columns:
    df["expos"] = 1

run_mvp_frequency_eda(df, EDA_CONFIG)

## Обучение моделей

Обучение вынесено в отдельный пакет `querulus.training`. Сборка датасета и обучение запускаются разными ячейками.

In [ ]:
import importlib

importlib.invalidate_caches()

from IPython.display import Markdown, display

from querulus.training.config import TrainingConfig
from querulus.training.pipeline import (
    format_features_table,
    format_metrics_table,
    format_training_summary,
    train_models,
)
# None = полный MVP-пул; frequency: + select_features при frequency_select_features=True.
FREQUENCY_FEATURES = None
SEVERITY_FEATURES = None

TRAINING_CONFIG = TrainingConfig(
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
    frequency_features=FREQUENCY_FEATURES,
    severity_features=SEVERITY_FEATURES,
    frequency_select_features=False,
    frequency_num_features_to_select=20,
    frequency_calibration_enabled=False,
)
training = train_models(df, TRAINING_CONFIG)

display(Markdown("### Training summary"))
display(format_training_summary(training.summary))

display(Markdown("### Frequency features"))
display(format_features_table(training.frequency_features, training.frequency_categorical_features))

display(Markdown("### Severity features"))
display(format_features_table(training.severity_features, training.severity_categorical_features))

display(Markdown("### Frequency (classification)"))
display(format_metrics_table(training.frequency_metrics_table))
display(Markdown("### Severity (regression)"))
display(format_metrics_table(training.severity_metrics_table))

display(Markdown("### Frequency importance"))
display(training.frequency_importance.head(30))
display(Markdown("### Severity importance"))
display(training.severity_importance.head(30))

## Диагностика модели

Распределение вероятностей классификации и ModelDiagnostics (после обучения).

In [ ]:
from querulus.training.plots import run_model_diagnostics_visualizations

run_model_diagnostics_visualizations(df, training, TRAINING_CONFIG)

## Финансовый эффект

Расчёт факта/модели, подбор порога классификации, сводная таблица и экспорт — пакет `querulus.fin_effect` (логика из Litigant `fin_effect.py`).

In [ ]:
from querulus.fin_effect import (
    create_summary_table,
    export_analytics_excel,
    export_summary_excel,
    print_best_threshold_report,
    resolve_fin_effect_config,
    run_fin_effect_from_training,
)

FIN_EFFECT_CONFIG = resolve_fin_effect_config(
    df,
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
    loaded_from_checkpoint=LOAD_FINAL_DATASET,
    legacy_dataset=LOAD_LEGACY_DATASET,
)
print(f"fin_effect fact_mode={FIN_EFFECT_CONFIG.fact_mode}")

FIN_EFFECT_DIR = PROJECT_ROOT / "data" / "processed" / "fin_effect"
FIN_EFFECT_DIR.mkdir(parents=True, exist_ok=True)

# test-сплит по дате (как в обучении); порог — max net_effect
fin_effect = run_fin_effect_from_training(
    df, training, split="test", config=FIN_EFFECT_CONFIG
)
print_best_threshold_report(fin_effect)

summary_table = create_summary_table(fin_effect.frame, FIN_EFFECT_CONFIG)
display(summary_table.style.format("{:,.0f}", subset=summary_table.columns[3:]))

export_summary_excel(summary_table, FIN_EFFECT_DIR / "summary_table.xlsx")
export_analytics_excel(
    fin_effect.frame, FIN_EFFECT_DIR / "Аналитика.xlsx", config=FIN_EFFECT_CONFIG
)
print(f"Экспорт: {FIN_EFFECT_DIR}")

In [ ]:
from querulus.fin_effect import (
    plot_confusion_matrix,
    plot_cost_confusion_heatmaps,
    plot_positive_cases_by_month,
    plot_precision_recall_vs_threshold,
    plot_severity_fact_vs_pred_binned,
    plot_target_monthly_share,
)

split = training.frequency_split
frame = fin_effect.frame
y_test = split.y_test.loc[frame.index]
x_test = split.x_test.loc[frame.index, training.frequency_features]

business_threshold = fin_effect.best_threshold
plot_confusion_matrix(
    frame[FIN_EFFECT_CONFIG.frequency_target_column],
    frame["pred_freq"],
    title=f"Confusion Matrix {FREQUENCY_TARGET} (threshold={business_threshold:.2f})",
)
plot_precision_recall_vs_threshold(training.frequency_model, x_test, y_test)
plot_cost_confusion_heatmaps(frame, y_test, frame["pred_freq"])
plot_severity_fact_vs_pred_binned(
    frame,
    frame[SEVERITY_TARGET],
    frame["pred_sev"],
    mask_actual_claim=(y_test == 1),
    config=FIN_EFFECT_CONFIG,
)
plot_target_monthly_share(df, config=FIN_EFFECT_CONFIG)
plot_positive_cases_by_month(df, config=FIN_EFFECT_CONFIG)